In [0]:
dbutils.widgets.text("catalog","")
CATALOG=dbutils.widgets.get("catalog")
dbutils.widgets.text("schema","")
SCHEMA=dbutils.widgets.get("schema")
dbutils.widgets.text("table","")
TABLE=dbutils.widgets.get("table")

In [0]:
spark.sql(f"create catalog if not exists {CATALOG};")
spark.sql(f"create schema if not exists {CATALOG}.{SCHEMA};")
#spark.sql(f"create volume if not exists {CATALOG}.{SCHEMA}.raw_data;")
spark.sql(f"create volume if not exists {CATALOG}.{SCHEMA}.bronze;")

In [0]:
%run /Workspace/Users/vibhacse@gmail.com/databricks/Delta_Practice/Generic_Functions/utils

In [0]:
drugdf1=read_csv(spark,"/Volumes/pharmaceutical/medicines/raw_data/druginfo.csv",True,True)
display(drugdf1)

In [0]:
write_delta_file(drugdf1,"delta","/Volumes/pharmaceutical/medicines/bronze/")

In [0]:
%skip
drugdf1.write.saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
%sql
use pharmaceutical.medicines;
desc history bronze_druginfo;
select * from bronze_druginfo;

In [0]:
%skip
%sql
--update query
update bronze_druginfo set rating=rating-1 where uniqueid=17957;
select * from bronze_druginfo where uniqueid=17957;

In [0]:
%sql
desc history bronze_druginfo;

In [0]:
%sql
select count(1) from bronze_druginfo;
--delete query
delete from bronze_druginfo
where uniqueid=215892;
--check count post delete
select count(1) from bronze_druginfo;

In [0]:
display(describe_delta_table('bronze_druginfo'))

In [0]:
%sql
--CTAS(create table as select)
--create or replace table bronze_druginfo_merge as select * from bronze_druginfo where rating<8;
select count(*) from bronze_druginfo_merge;
select count(*) from bronze_druginfo;
--select * from bronze_druginfo_merge;
--update bronze_druginfo_merge set condition='Hyper' where uniqueid=6971;
--select * from bronze_druginfo_merge where uniqueid=6971;
--select * from bronze_druginfo where uniqueid=6971;
--insert into bronze_druginfo_merge (uniqueid,drugname,condition,rating,date,usefulcount) values (1100,'supernova','good',8,'2012-12-13',20);
select * from bronze_druginfo_merge where uniqueid=1100;
select count(*) from bronze_druginfo_merge;

In [0]:
%sql
--Merge of two tables - target > "bronze_druginfo_merge" and source > "bronze_druginfo"
--Update if records exist in target, Insert if records are present in source and Delete records from target if not present in source
merge into bronze_druginfo_merge tgt
using bronze_druginfo src
on tgt.uniqueid=src.uniqueid
when matched then
  update set
    tgt.drugname=src.drugname,
    tgt.condition=src.condition,
    tgt.usefulcount=src.usefulcount
when not matched then
  insert (tgt.uniqueid,tgt.drugname,tgt.condition,tgt.rating,tgt.date,tgt.usefulcount)values(src.uniqueid,src.drugname,src.condition,src.rating,src.date,src.usefulcount)
when not matched by source
  then delete;

In [0]:
display(describe_delta_table('bronze_druginfo_merge'))